[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-Music3_Colab.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-Music3_Colab.ipynb)

# 🎵 MiniMax-Music3 — Lyrics + Caption-to-Music (Qwen3 + Flow-Matching DiT + Flow-VAE)
> **Runtime required:** GPU. **Recommended:** **L4 (24 GB)** for the bf16 path with the LM kept resident, **A100 (40/80 GB)** for full quality with no offload. T4 (16 GB) needs CPU offload — slow but workable.
A Colab port of [MiniMax-Music3](https://huggingface.co/MiniMaxAI/MiniMax-Music3) — a lyrics- and caption-conditioned music generation model that produces **stereo audio at 44.1 kHz** with vocals, instruments, and structure. Built on the official [diffusers `MiniMaxMusic3ModularPipeline`](https://github.com/huggingface/diffusers/tree/main/src/diffusers/modular_pipelines/minimax_music3) (merged into `main` 2026-08-13).
## How it works
MiniMax-Music3 is a **three-stage pipeline**:
1. **Semantic generation** — Qwen3-8B language model + RVQ depth decoder reads the structured caption (genre / mood / vocals / instrumentation / arrangement) + tagged lyrics (`[verse]`, `[chorus]`, `[instrumental]`, ...) and autoregressively generates per-frame semantic tokens + hidden states (25 Hz AR frame rate).
2. **Core denoise** — 2.4B flow-matching transformer runs 30 Euler steps per 200-frame window (with 100-frame overlap) to generate the 128-channel Flow-VAE audio latent from noise, conditioned on the LM's hidden states.
3. **Vocoder decode** — 123M Flow-VAE vocoder turns the audio latents into a stereo waveform at **44.1 kHz**. Windows are stitched via overlap-add (drops leading 86 / trailing 344-86 latent frames per window).
```
prompt + lyrics -> Qwen3-8B + RVQ depth decoder -> per-frame hidden states
                                          |
                                          v
                          Flow-Matching DiT (2.4B, 30 steps/window)
                                          |
                                          v
                                  Flow-VAE latent (128-ch)
                                          |
                                          v
                                Flow-VAE vocoder (123M)
                                          |
                                          v
                                  stereo WAV @ 44.1 kHz
```
## ⚠️ License
Released under the **MiniMax Music 3 Community License**. Check the model card before commercial use.
## Quick start
1. **Runtime → Change runtime type → GPU** (L4, A100, or A100 80GB)
2. Run **STEP 1** — installs torch 2.11.0+cu128, diffusers from `main` (MiniMax-Music3 is in `main` as of 2026-08-13), torchaudio, scipy. First run: ~5-10 min.
3. Run **STEP 2** — downloads the 7-component modular checkpoint (~27 GB total: 16 GB Qwen3-8B + 9 GB DiT + 1.2 GB RVQ decoder + 0.5 GB misc). First run: ~15-30 min.
4. Run **STEP 3** — imports + lazy pipe loader. Loads the pipe once into VRAM (~24 GB bf16 at rest).
5. Run **STEP 4** for the Gradio UI, or **STEP 6** for a single-song quick test, or **STEP 7** for batch.
6. **STEP 5** keep-alive prevents Colab from disconnecting.
## Memory
| GPU | VRAM | Mode | Peak VRAM | 60s song runtime |
|-----|------|------|-----------|------------------|
| **A100 80GB** | 80 GB | `bf16` (full) | ~26 GB | ~2-3 min |
| **L4 24GB** | 24 GB | `bf16` (full, tight) | ~24 GB | ~5-7 min |
| **L4 24GB** | 24 GB | `cpu_offload` | ~10 GB peak (LM streamed off) | ~10-15 min |
| **T4 16GB** | 16 GB | `cpu_offload` (forced) | ~10 GB peak | ~15-20 min |
The notebook auto-detects GPU VRAM and recommends a mode. The `cpu_offload` mode trades time for VRAM — usable on T4 but slow.
## Outputs
Each generation produces a **stereo WAV file** at 44.1 kHz (`*.wav` in `AEI_3D_Out/MiniMax-Music3/songs/`). The file is **float32 in `[-1, 1]`** written as **PCM int16** for compatibility with all audio editors.
Default song length is 60 seconds. Max is 360 seconds (6 min, capped at 9000 acoustic frames).

In [ ]:
#@title STEP 1 — Install torch, diffusers main, torchaudio, scipy
"""
• Pins torch to 2.11.0+cu128 (recent enough for the MiniMax-Music3 modular pipeline)
• Installs diffusers from main (MiniMax-Music3 is merged as of 2026-08-13, no PR pin needed)
• Installs torchaudio (for audio utilities)
• Installs scipy (for WAV writing via scipy.io.wavfile.write)
• Stubs the `spaces` module (HF ZeroGPU API, not available on Colab)
• Mounts Google Drive for checkpoint caching
"""
import os, sys, time, subprocess, pathlib
print('='*72)
print('MiniMax-Music3 - Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()
CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/MiniMax-Music3')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Drive cache  : {drive_root}')
else:
    drive_root = pathlib.Path('/content/_mm_music3_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}')
OUT_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/MiniMax-Music3')
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
# --- 1. Install / upgrade torch ---
t0 = time.time()
print('\n[1/4] Ensuring torch 2.11.0+cu128 ...')
try:
    import torch
    if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 11):
        print(f'  Upgrading torch {torch.__version__} -> 2.11.0+cu128 ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                        'torch>=2.11.0,<2.12.0',
                        '--index-url', 'https://download.pytorch.org/whl/cu128'],
                       check=True)
    else:
        print(f'  torch {torch.__version__} OK')
except ImportError:
    print('  Installing torch 2.11.0+cu128 ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch>=2.11.0,<2.12.0',
                    '--index-url', 'https://download.pytorch.org/whl/cu128'],
                   check=True)
print(f'  torch ready in {time.time()-t0:.1f}s')
# --- 2. Install diffusers from main (MiniMax-Music3 is in MODULAR_PIPELINE_MAPPING) ---
t0 = time.time()
print('\n[2/4] Installing diffusers from main ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                'git+https://github.com/huggingface/diffusers.git',
                '--upgrade'],
               check=True)
print(f'  diffusers installed in {time.time()-t0:.1f}s')
# --- 3. Install supporting deps ---
t0 = time.time()
print('\n[3/4] Installing supporting deps (torchaudio, scipy, soundfile) ...')
EXTRA_PKGS = [
    'torchaudio',
    'scipy',
    'soundfile',
    'librosa',
    'tqdm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_PKGS, check=False)
print(f'  extra deps ready in {time.time()-t0:.1f}s')
# --- 4. Stub the `spaces` module (HF ZeroGPU API, not available on Colab) ---
t0 = time.time()
print('\n[4/4] Stubbing `spaces` module ...')
import types as _types
if 'spaces' not in sys.modules:
    _spaces_stub = _types.ModuleType('spaces')
    def _noop_decorator(*args, **kwargs):
        # Handle both @spaces.GPU and @spaces.GPU(duration=...) forms
        if len(args) == 1 and callable(args[0]) and not kwargs:
            return args[0]
        def _decorator(fn):
            return fn
        return _decorator
    _spaces_stub.GPU = _noop_decorator
    _spaces_stub.CUDA = _noop_decorator
    _spaces_stub.aoti_load_from_package_dir = lambda *a, **kw: None
    sys.modules['spaces'] = _spaces_stub
    print('  spaces stub registered (no-op)')
else:
    print('  spaces already loaded (likely from previous run)')
print(f'  stub setup in {time.time()-t0:.1f}s')
# --- Verify imports ---
print('\nVerifying imports ...')
try:
    import diffusers
    print(f'  diffusers     : {diffusers.__version__}')
except Exception as e:
    print(f'  [FAIL] diffusers: {e}')
    raise
try:
    from diffusers import ModularPipeline
    print('  ModularPipeline: importable')
except Exception as e:
    print(f'  [FAIL] ModularPipeline: {e}')
    raise
try:
    from diffusers.modular_pipelines.minimax_music3 import MiniMaxMusic3ModularPipeline
    print('  MiniMaxMusic3ModularPipeline: importable')
except Exception as e:
    print(f'  [FAIL] MiniMaxMusic3ModularPipeline: {e}')
    raise
try:
    from diffusers.models import MiniMaxMusic3Transformer1DModel
    print('  MiniMaxMusic3Transformer1DModel: importable')
except Exception as e:
    print(f'  [WARN] model class: {e}')
try:
    import torchaudio, scipy.io.wavfile, soundfile, librosa
    print('  torchaudio, scipy, soundfile, librosa: OK')
except Exception as e:
    print(f'  [FAIL] audio deps: {e}')
    raise
print('\n' + '='*72)
print('STEP 1 done. Next: run STEP 2 to download weights (~27 GB).')
print('='*72)

In [ ]:
#@title STEP 4 — (Optional) Gradio UI for MiniMax-Music3
"""
Launches a Gradio app with structured caption fields + lyrics editor + audio player.
The pipe loads lazily on the first Generate click (so the cell returns immediately).
The UI mirrors the official MiniMax Music 3 Space at https://huggingface.co/spaces/MiniMaxAI/MiniMax-Music3
but without ZeroGPU / aoti_load_from_package_dir (those are HF-Spaces-only).
The lyrics field is a multi-line textbox - one structure tag per line, e.g.:
    [verse]
    Walking through the city lights
    Every shadow tells a story
    [chorus]
    We are the night, we are the light
For instrumental music, just put `[instrumental]` on the first line.
"""
import os, sys, time, json, random, builtins
from pathlib import Path
import gradio as gr
import numpy as np
import torch
# Pull generate_song from builtins (exposed by STEP 3)
_generate_song = builtins._generate_song
OUT_DIR = builtins.MiniMax_MUSIC3_OUT_DIR
DEFAULT_CAPTION = (
    'Global Metadata -\n'
    '  Genre: Synthwave\n'
    '  BPM: 100\n'
    '  Key: A minor\n'
    '  Mood: Nostalgic, cinematic\n'
    'Vocal Details -\n'
    '  Gender: Male\n'
    '  Timbre: Warm baritone\n'
    '  Vocal style: Soft, breathy, slightly reverbed\n'
    'Arrangement -\n'
    '  Intro: Soft synth pad + reverb swell, no drums\n'
    '  Verse: Kick + bass drop in, arpeggiated synth carries the rhythm\n'
    '  Chorus: Full band, lead synth melody on top, layered pads\n'
    '  Bridge: Strip back to vocals + piano, builds back to final chorus'
)

DEFAULT_LYRICS = (
    '[verse]\n'
    'Lights are flickering on the boulevard tonight\n'
    'Neon signs reflect in puddles from the afternoon rain\n'
    '[chorus]\n'
    'We chase the shadows down the avenue\n'
    'Finding pieces of ourselves we never knew\n'
    '[verse]\n'
    'Strangers pass like ghosts in colored light\n'
    'Every face a story we will never write\n'
    '[chorus]\n'
    'We chase the shadows down the avenue\n'
    'Finding pieces of ourselves we never knew\n'
    '[outro]\n'
    '(instrumental fade)'
)
def _run(prompt, lyrics, duration, steps, seed, randomize_seed, request: gr.Request):
    if randomize_seed or seed == 0:
        seed = random.randint(1, 2**31 - 1)
    try:
        result = _generate_song(
            prompt=prompt,
            lyrics=lyrics,
            audio_duration=float(duration),
            num_inference_steps=int(steps),
            seed=int(seed),
            save=True,
            prefix='gradio',
        )
        return (result['wav_path'], result['seed_used'], result['duration_s'])
    except Exception as e:
        raise gr.Error(f'{type(e).__name__}: {e}')
    gr.Markdown(
        '## Welcome to MiniMax-Music3\n\n'
        'Enter a **structured caption** (or free-form description) and **tagged lyrics** above.\n'
        'Structure tags like `[verse]`, `[chorus]`, `[instrumental]` must each be on their own line.\n\n'
        'The pipe loads lazily on the first Generate click (30-60s init). Each song takes 2-15 min.\n\n'
        'Use the **Randomize seed** checkbox to try different generations of the same prompt.\n\n'
        'Default caption + lyrics below are a lo-fi / indie-pop example - replace with your own.'
    )
with gr.Blocks(title='MiniMax-Music3', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎵 MiniMax-Music3\nLyrics- and caption-conditioned music generation. 44.1 kHz stereo.')
    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(value=DEFAULT_CAPTION, label='Structured Caption',
                                lines=10, info='Genre/mood/vocals/instrumentation/arrangement. Free-form text is fine too.')
            lyrics = gr.Textbox(value=DEFAULT_LYRICS, label='Lyrics (one structure tag per line)',
                                lines=12, info='[verse]/[chorus]/[bridge]/[instrumental]/[solo]/[intro]/[outro] etc.')
        with gr.Column(scale=1):
            duration = gr.Slider(minimum=10, maximum=300, value=60, step=5, label='Duration (seconds)')
            steps = gr.Slider(minimum=10, maximum=100, value=30, step=1, label='Inference steps (per chunk)')
            seed = gr.Number(value=0, label='Seed (0 = random)', precision=0)
            randomize_seed = gr.Checkbox(value=True, label='Randomize seed')
            btn = gr.Button('Generate', variant='primary')
            out_audio = gr.Audio(label='Generated song', type='filepath')
            out_seed = gr.Textbox(label='Seed used', interactive=False)
            out_duration = gr.Textbox(label='Duration (s)', interactive=False)
    def _show_welcome():
        return '**MiniMax-Music3 ready.** Click Generate to start. The pipe loads on first use (~30-60s).'
    demo.load(_show_welcome, inputs=None, outputs=None)
    btn.click(
        fn=_run,
        inputs=[prompt, lyrics, duration, steps, seed, randomize_seed],
        outputs=[out_audio, out_seed, out_duration],
        api_name='generate',
    )
# Cell 5 launch — use share=False for Colab (we don't need a public URL).
# concurrency_limit=1 since each gen takes 2-15 minutes and the pipe is a single global.
# clear_output() keeps the cell from filling up with status prints.
from IPython.display import display, clear_output
clear_output()
demo.queue(concurrency_limit=1).launch(share=False, inline=False, prevent_thread_lock=True, quiet=True)
display(demo)
print('STEP 4 launched. The Gradio UI is interactive in the output above.')
print('Each generation takes 2-15 min depending on GPU + duration. The pipe is shared, so concurrent requests are serialized.')

In [ ]:
#@title STEP 5 — Keep Colab alive + session summary
"""
Prevents Colab from disconnecting by polling a heartbeat (any heavy import works).
Also prints a summary of the current session state.
"""
import time, json, os
from pathlib import Path
# Session summary
try:
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        vram_gb = p.total_memory / 1024**3
        print(f'GPU: {p.name} ({vram_gb:.1f} GB)')
    else:
        print('GPU: none')
except Exception:
    print('GPU: unavailable')
try:
    import diffusers
    print(f'diffusers: {diffusers.__version__}')
except Exception:
    print('diffusers: not installed')
try:
    repo_dir = builtins.MiniMax_MUSIC3_REPO_DIR()
    print(f'Checkpoint: {repo_dir}')
    # Total size
    total = sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, files in os.walk(repo_dir) for f in files
    )
    print(f'  Total on disk: {total / 1024**3:.2f} GB')
except Exception:
    print('Checkpoint: not found (re-run STEP 2)')
try:
    out_dir = builtins.MiniMax_MUSIC3_OUT_DIR
    wavs = sorted(out_dir.glob('*.wav'))
    print(f'Output dir : {out_dir}')
    print(f'  songs: {len(wavs)}')
    total_audio_s = 0
    for w in wavs[-5:]:
        import scipy.io.wavfile as _wav
        sr, audio = _wav.read(str(w))
        secs = audio.shape[0] / sr
        total_audio_s += secs
        print(f'    {secs:.1f}s  {w.name}')
except Exception as e:
    print(f'Output dir: {e}')
# Heartbeat loop (re-run STEP 5 to interrupt)
print('\nHeartbeat loop (Ctrl+C or Interrupt to stop):')
try:
    while True:
        time.sleep(60)
        print(f'  ... alive at {time.strftime("%H:%M:%S")}')
except KeyboardInterrupt:
    print('Heartbeat stopped.')

In [ ]:
#@title STEP 6 — Quick test (single song generation)
"""
Generate one song with form-widget inputs and immediately play it. Useful for
verifying the pipeline works after install + download.
The pipe is loaded lazily on first call (STEP 3 init). Expect ~30-60s
initialization plus the generation time (~2-7 min for 60s of audio).
"""
import time, json, random, builtins
from pathlib import Path
from IPython.display import display, Audio, FileLink
PROMPT = (
    'Global Metadata -\n'
    '  Genre: Lo-fi hip hop\n'
    '  BPM: 85\n'
    '  Key: C major\n'
    '  Mood: Relaxed, study-friendly, warm\n'
    'Vocal Details -\n'
    '  Gender: Female\n'
    '  Timbre: Soft alto with gentle vibrato\n'
    '  Vocal style: Whispered, intimate\n'
    'Arrangement -\n'
    '  Intro: 4 bars of vinyl crackle + soft piano chord\n'
    '  Verse: Mellow kick + sub bass, jazzy Rhodes chords\n'
    '  Chorus: Add brushed snare + warm string pad, vocal doubles\n'
    '  Bridge: Strip to piano + vocal, then build back to final chorus'
)

LYRICS = (
    '[intro]\n'
    '(instrumental)\n'
    '[verse]\n'
    'Coffee steam is curling up against the windowpane\n'
    'Notes scattered on the desk like fallen leaves again\n'
    '[chorus]\n'
    'Let the morning keep on moving slow\n'
    'There is nowhere else I need to go\n'
    '[verse]\n'
    'Pages turning like the hands upon a quieter clock\n'
    'Every shadow dancing where the sun begins to walk\n'
    '[chorus]\n'
    'Let the morning keep on moving slow\n'
    'There is nowhere else I need to go\n'
    '[outro]\n'
    '(instrumental fade)'
)
DURATION = 60  #@param {type:'slider', min:10, max:300, step:5}
STEPS = 30  #@param {type:'slider', min:10, max:100, step:1}
SEED = 0  #@param {type:'integer'}

print(f'  Prompt  : {PROMPT[:60]}...')
print(f'  Lyrics  : {LYRICS.splitlines()[0] if LYRICS else ""}...')
print(f'  Duration: {DURATION}s, steps={STEPS}, seed={SEED} (0 = random)')
print()

result = builtins._generate_song(
    prompt=PROMPT,
    lyrics=LYRICS,
    audio_duration=DURATION,
    num_inference_steps=STEPS,
    seed=SEED,
    save=True,
    prefix='quicktest',
)

print(f'  Done: {result["duration_s"]:.1f}s of audio, seed={result["seed_used"]}')
print(f'  File: {result["wav_path"]}')
print()
display(Audio(result['wav_path']))
try:
    display(FileLink(result['wav_path']))
except Exception:
    pass


In [ ]:
#@title STEP 7 — Batch generation from a JSON scene list
"""
Reads a JSON file containing a list of songs and generates them sequentially.
The pipe is loaded once and reused across all entries. Per-entry skip-existing
with hash-based change detection (same scheme as MiniMax-H3 + LTX-2.5 notebooks).
JSON format (a list of objects) -
```json
[
  {
    "prompt": "Global Metadata: Genre: Synthwave, BPM: 110, ...",
    "lyrics": "[verse]\nLyrics line 1\n[chorus]\nLyrics line 2",
    "duration": 60,
    "steps": 30,
    "seed": 42
  },
  {
    "prompt": "..."
    "lyrics": "[instrumental]"
    "duration": 30,
    "seed": 123
  }
]
```
Fields (only `prompt` and `lyrics` are required; rest override the form-widget defaults):
  - prompt:    (required) structured caption OR free-form music description.
  - lyrics:    (required) lyrics with [verse]/[chorus]/[instrumental] tags, one per line.
  - duration:  (optional, default 60) seconds (1-300, capped at 9000 latent frames ~ 6 min).
  - steps:     (optional, default 30) flow-matching Euler steps per chunk (10-100).
  - seed:      (optional, default 0 = random) int seed for reproducibility.
The file at BATCH_JSON_PATH is created with a starter template if it does not exist.
"""
import os, sys, time, json, random, hashlib, builtins
from pathlib import Path
BATCH_JSON_PATH = '/content/drive/MyDrive/AEI_3D_Cache/MiniMax-Music3/batch_songs.json'  #@param {type:'string'}
DEFAULT_DURATION = 60  #@param {type:'slider', min:10, max:300, step:5}
DEFAULT_STEPS    = 30  #@param {type:'slider', min:10, max:100, step:1}
SKIP_EXISTING = True  #@param {type:'boolean'}
RESUME_FROM_LOG = True  #@param {type:'boolean'}
batch_path = Path(BATCH_JSON_PATH)
if not batch_path.exists():
    batch_path.parent.mkdir(parents=True, exist_ok=True)
    starter = [
        {
            'prompt': 'Global Metadata: Genre: Ambient electronic, BPM: 90, Key: D minor, Mood: Reflective, introspective.\nVocal Details: No vocals (instrumental).\nArrangement: Slow build with pads, sparse piano, no percussion.',
            'lyrics': '[instrumental]\n(soft pad swells)\n(gentle piano motif repeats)\n(careful layering of textures)\n(gentle fade)',
            'duration': 60,
            'steps': 30,
            'seed': 0
        },
        {
            'prompt': 'Global Metadata: Genre: Indie folk, BPM: 95, Key: G major, Mood: Uplifting, hopeful.\nVocal Details: Gender: Male, Timbre: Warm tenor, Vocal style: Bright, clear.\nArrangement: Acoustic guitar arpeggios, light brushed drums, layered harmonies on chorus.',
            'lyrics': '[verse]\nWaking up to morning sun through an open window\nThe world outside is waking too\n[chorus]\nEvery day a chance to start again\nEvery step a door that opens from within\n[verse]\nCoffee steam and quiet thoughts in a kitchen chair\nThe simple things that bring us there\n[chorus]\nEvery day a chance to start again\nEvery step a door that opens from within',
            'duration': 60,
            'steps': 30,
            'seed': 0
        }
    ]
    batch_path.write_text(json.dumps(starter, indent=2))
    print(f'  Wrote starter batch: {batch_path}')
    print('  Edit it (add prompt, lyrics, duration, steps, seed), then re-run STEP 7.')
else:
    print(f'  Using existing batch JSON: {batch_path}')
scenes = json.loads(batch_path.read_text())
if not isinstance(scenes, list):
    raise SystemExit(f'Batch JSON must be a list, got {type(scenes).__name__}')
print(f'\n  Scenes in batch: {len(scenes)}')
# Resume log (one JSON line per scene, append-only)
_log_path = builtins.MiniMax_MUSIC3_OUT_DIR / 'batch_songs.log.jsonl'
_completed = set()
if RESUME_FROM_LOG and _log_path.exists():
    for line in _log_path.read_text().splitlines():
        try:
            entry = json.loads(line)
            if entry.get('status') == 'ok':
                _completed.add(entry['index'])
        except Exception:
            pass
    print(f'  Resume log: {len(_completed)} already-completed')
def _song_prompt_hash(prompt, lyrics, duration, steps):
    h = hashlib.sha256()
    h.update(prompt.encode('utf-8'))
    h.update(b'\x00')
    h.update(lyrics.encode('utf-8'))
    h.update(b'\x00')
    h.update(str(duration).encode())
    h.update(b'\x00')
    h.update(str(steps).encode())
    return h.hexdigest()[:16]
results = []
total_start = time.time()
for i, sc in enumerate(scenes):
    if i in _completed:
        print(f'  [{i+1}/{len(scenes)}] SKIP (resume log)')
        results.append(None)
        continue
    prompt = (sc.get('prompt') or '').strip()
    lyrics = (sc.get('lyrics') or '').strip()
    if not prompt:
        print(f'  [{i+1}/{len(scenes)}] SKIP: empty prompt')
        results.append(None)
        continue
    if not lyrics:
        print(f'  [{i+1}/{len(scenes)}] SKIP: empty lyrics')
        results.append(None)
        continue
    duration = int(sc.get('duration', DEFAULT_DURATION))
    steps = int(sc.get('steps', DEFAULT_STEPS))
    seed = int(sc.get('seed', 0))
    hash_id = _song_prompt_hash(prompt, lyrics, duration, steps)
    existing = sorted(builtins.MiniMax_MUSIC3_OUT_DIR.glob(f'*_seed*_{hash_id[:8]}_*.wav'))
    if SKIP_EXISTING and existing:
        print(f'  [{i+1}/{len(scenes)}] SKIP (hash match): {existing[0].name}')
        results.append(str(existing[0]))
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'ok', 'path': str(existing[0]),
                                'hash': hash_id, 'skipped': True}) + chr(10))
        continue
    print(f'\n  [{i+1}/{len(scenes)}] d={duration}s steps={steps} seed={seed} hash={hash_id[:8]}')
    print(f'    prompt: {prompt[:60]}...')
    print(f'    lyrics: {lyrics[:60].splitlines()[0] if lyrics else ""}...')
    try:
        t0 = time.time()
        result = builtins._generate_song(
            prompt=prompt,
            lyrics=lyrics,
            audio_duration=duration,
            num_inference_steps=steps,
            seed=seed,
            save=True,
            prefix=f'batch{i+1:03d}',
        )
        elapsed = time.time() - t0
        print(f'    -> {elapsed:.0f}s, {result["duration_s"]:.1f}s audio')
        results.append(result['wav_path'])
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'ok',
                                'path': result['wav_path'],
                                'duration_s': result['duration_s'],
                                'elapsed_s': elapsed,
                                'seed': result['seed_used'],
                                'hash': hash_id}) + chr(10))
    except Exception as e:
        print(f'    FAIL: {type(e).__name__}: {e}')
        results.append(None)
        with _log_path.open('a') as f:
            f.write(json.dumps({'index': i, 'status': 'fail',
                                'error': f'{type(e).__name__}: {e}'}) + chr(10))
# Batch summary
total = time.time() - total_start
ok_count = sum(1 for r in results if r is not None)
fail_count = sum(1 for r in results if r is None and i < len(scenes))
print('\n' + '='*72)
print(f'  BATCH SUMMARY')
print('='*72)
print(f'  Scenes:  {ok_count} succeeded, {len(scenes) - ok_count} failed/skipped, {len(scenes)} total')
print(f'  Total:   {_fmt_seconds(total)}')
print(f'  Log:     {_log_path}')
print()
print('  Output files:')
for r in results:
    if r:
        print(f'    {r}')
print('='*72)
def _fmt_seconds(seconds):
    s = int(seconds); h, rem = divmod(s, 3600); m, ss = divmod(rem, 60)
    return f'{h}:{m:02d}:{ss:02d}' if h else f'{m:02d}:{ss:02d}'

In [ ]:
#@title STEP 8 — Tail the latest log / errors (debug aid)
"""
Prints the tail of the batch log and any error messages. Useful when a generation
fails or you want to see which scenes completed.
"""
import json
from pathlib import Path
_log_path = builtins.MiniMax_MUSIC3_OUT_DIR / 'batch_songs.log.jsonl'
TAIL_LINES = 30  #@param {type:'slider', min:5, max:200, step:5}
if not _log_path.exists():
    print(f'No log file yet at {_log_path}')
    print('Run STEP 7 to generate songs (creates the log).')
else:
    lines = _log_path.read_text().splitlines()
    tail = lines[-TAIL_LINES:]
    print(f'Tail of {_log_path.name} ({len(lines)} total entries):\n')
    for line in tail:
        try:
            entry = json.loads(line)
            ts = entry.get('index', '?')
            st = entry.get('status', '?')
            extras = []
            if 'path' in entry:
                extras.append(f'path={Path(entry["path"]).name}')
            if 'duration_s' in entry:
                extras.append(f'duration={entry["duration_s"]:.1f}s')
            if 'elapsed_s' in entry:
                extras.append(f'elapsed={entry["elapsed_s"]:.0f}s')
            if 'seed' in entry:
                extras.append(f'seed={entry["seed"]}')
            if 'error' in entry:
                extras.append(f'error={entry["error"]}')
            extras_str = ' '.join(extras)
            print(f'  [{ts:>3}] {st:5s} {extras_str}')
        except Exception:
            print(f'  (raw) {line[:200]}')